# Per-region contributions to the LFADS latent space — kcEXP00H

This notebook answers: **which brain regions drive which latent factors, and how strongly?**

The multisession LFADS architecture maps 75 regions → 50 shared factors via two linear layers:
- **Readin** (frozen, PCR-init): `x_t (n_regions,) → z_t (50,)` — what the encoder sees
- **Readout** (trained): `f_t (50,) → x̂_t (n_regions,)` — how factors reconstruct each region

The readout weight matrix `W` (`[n_regions × n_factors]`) is the central object here:
- `W[i, j]` = how strongly factor `j` drives region `i`
- It is interpretable because reconstruction is linear: `x̂_t ≈ W @ f_t + b`

**Requires:** trained model output — run `scripts/run_pbt.py` (or `run_multi.py`) first.

Analyses:
1. Recover `W_readout` from HDF5 posterior output (OLS) or from model checkpoint
2. Readout weight heatmap — regions × factors
3. Factor-region correlation from posterior means
4. Variance contribution per region per factor
5. Readin weight comparison (encoder side, from checkpoint)
6. Cross-fly consistency of region loadings

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import h5py
from glob import glob
from pathlib import Path

sys.path.insert(0, '/media/server/gklab/KarenCheng/DATA/LFM_current/kcEXP00H/kcEXP00H_PythonNotebooks')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9

## Path configuration

Set `H5_DIR` to the directory containing the `lfads_output_*.h5` files produced after training.
Set `CKPT_PATH` to a model checkpoint (`.ckpt`) if you want exact readout weights from the trained model;
set to `None` to infer weights by regression from the HDF5 data instead.

In [ ]:
XARRAY_DIR = (
    '/media/server/gklab/KarenCheng/DATA/LFM_current/'
    'kcEXP00H/kcEXP00H_PythonNotebooks/xarray_files/'
)

# Directory containing lfads_output_*.h5 files (from run_posterior_sampling)
H5_DIR = None   # e.g. Path('../runs/kcEXP00H_pbt/run_model_xxxx_00003')

# Optional: path to best model checkpoint for exact weight extraction
CKPT_PATH = None  # e.g. Path('../runs/.../checkpoints/last.ckpt')

## Step 1 — Region names from xarray

In [ ]:
from utils.Xarray_UtilFns import load_group_xarrays

ds_list = load_group_xarrays(XARRAY_DIR)

# Canonical 75-region order (same across all flies in the xarray files)
REGION_NAMES = list(ds_list[0].coords['region'].values)
FLY_IDS = [ds.attrs['fly_id'] for ds in ds_list]
N_REGIONS = len(REGION_NAMES)

print(f'{N_REGIONS} regions: {REGION_NAMES[:8]}...')
print(f'Flies: {FLY_IDS}')

## Step 2 — Load HDF5 posterior outputs

Each `lfads_output_*.h5` file corresponds to one fly and contains
`train_factors`, `train_output_params`, `valid_factors`, `valid_output_params`, etc.

In [ ]:
assert H5_DIR is not None, "Set H5_DIR to the directory containing lfads_output_*.h5 files."

h5_paths = sorted(glob(str(Path(H5_DIR) / 'lfads_output_*.h5')))
print(f'Found {len(h5_paths)} HDF5 files:')
for p in h5_paths:
    print(f'  {Path(p).name}')

# Load all splits combined (train + valid) for each fly
fly_data = {}  # fly_id -> dict with 'factors', 'output_params'
for path in h5_paths:
    fly_stem = Path(path).stem.replace('lfads_output_', '')
    with h5py.File(path) as f:
        parts = {}
        for split in ('train', 'valid'):
            if f'{split}_factors' in f:
                parts[split] = {
                    'factors':       f[f'{split}_factors'][()],       # (n_trials, n_time, n_fac)
                    'output_params': f[f'{split}_output_params'][()], # (n_trials, n_time, n_reg)
                    'encod_data':    f[f'{split}_encod_data'][()],    # (n_trials, n_time, n_reg)
                }
        # Concatenate splits along trial axis
        factors = np.concatenate([parts[s]['factors'] for s in parts], axis=0)
        output_params = np.concatenate([parts[s]['output_params'] for s in parts], axis=0)
        fly_data[fly_stem] = {'factors': factors, 'output_params': output_params}
        print(f'  {fly_stem}: factors {factors.shape}, output_params {output_params.shape}')

## Step 3 — Recover readout weight matrix W

**From checkpoint (exact):** load `model.readout[i].weight` — shape `[n_regions, n_factors]`.

**From HDF5 by OLS (fallback):** because reconstruction is linear
(`output_params ≈ factors @ W.T + b`), we can recover W by regressing each region's
denoised activity onto all factors. This gives the same answer as the trained weights
(modulo very minor floating-point differences).

In [ ]:
from numpy.linalg import lstsq

def recover_readout_weights(factors, output_params):
    """OLS: output_params ~ factors @ W.T + b. Returns W [n_reg, n_fac] and b [n_reg]."""
    T, n_t, n_fac = factors.shape
    _, _, n_reg = output_params.shape
    F = factors.reshape(-1, n_fac)             # (T*n_t, n_fac)
    O = output_params.reshape(-1, n_reg)       # (T*n_t, n_reg)
    F_aug = np.column_stack([F, np.ones(len(F))])  # add intercept
    coef, *_ = lstsq(F_aug, O, rcond=None)    # (n_fac+1, n_reg)
    W = coef[:n_fac, :].T                     # (n_reg, n_fac)
    b = coef[n_fac, :]                        # (n_reg,)
    return W, b

# --- Option A: from model checkpoint ---
readout_weights = {}   # fly_stem -> W [n_reg, n_fac]
readin_weights  = {}   # fly_stem -> W_in [n_fac, n_reg]

if CKPT_PATH is not None:
    import torch
    sys.path.insert(0, str(Path(__file__).parents[1]))
    from lfads_torch.model import LFADS
    model = LFADS.load_from_checkpoint(CKPT_PATH)
    model.eval()
    fly_stems = list(fly_data.keys())
    for i, fly_stem in enumerate(fly_stems):
        W = model.readout[i].weight.detach().cpu().numpy()   # [n_reg, n_fac]
        readout_weights[fly_stem] = W
        W_in = model.readin[i].weight.detach().cpu().numpy() # [n_fac, n_reg]
        readin_weights[fly_stem] = W_in
    print('Readout weights loaded from checkpoint.')
else:
    # Option B: infer from HDF5 via OLS
    for fly_stem, d in fly_data.items():
        W, _ = recover_readout_weights(d['factors'], d['output_params'])
        readout_weights[fly_stem] = W
    print('Readout weights estimated by OLS from posterior means.')
    print('  (Set CKPT_PATH for exact weights; also required for readin weight analysis.)')

# Verify shape
for fly_stem, W in readout_weights.items():
    print(f'  {fly_stem}: W_readout {W.shape}  (n_regions × n_factors)')

## Step 4 — Match regions to canonical order

The HDF5 files preserve the 75-region order from the xarray data prep.
Regions with all-zero output_params (never-valid regions filled with 0) are flagged.

In [ ]:
# Flag regions that have near-zero variance in output_params (imputed zeros)
valid_mask = {}  # fly_stem -> bool array (n_regions,)
for fly_stem, d in fly_data.items():
    var_per_region = d['output_params'].var(axis=(0, 1))  # (n_regions,)
    valid_mask[fly_stem] = var_per_region > 1e-10
    n_valid = valid_mask[fly_stem].sum()
    print(f'  {fly_stem}: {n_valid}/{N_REGIONS} regions with non-zero variance')

## Fig 1 — Readout weight heatmap (regions × factors)

`W[i, j]` = weight from factor `j` to region `i`.
Strong positive/negative values → factor `j` strongly drives region `i`.

Regions are sorted by anatomical laterality (right-hemisphere then left);
factors are ordered as in the model (rotation-ambiguous — consider rotating to
interpretable axes, e.g., odour/walking/visual condition directions, before reading off indices).

In [ ]:
n_flies = len(readout_weights)
fly_stems = list(readout_weights.keys())
n_fac = list(readout_weights.values())[0].shape[1]

fig, axes = plt.subplots(1, n_flies, figsize=(4 * n_flies, 10), sharey=True)
if n_flies == 1:
    axes = [axes]

for ax, fly_stem in zip(axes, fly_stems):
    W = readout_weights[fly_stem]   # (n_regions, n_factors)
    # Normalize each factor column to unit max-abs for visual comparison
    W_norm = W / (np.abs(W).max(axis=0, keepdims=True) + 1e-12)
    
    im = ax.imshow(
        W_norm,
        aspect='auto', origin='lower',
        cmap='RdBu_r', vmin=-1, vmax=1,
        interpolation='nearest'
    )
    ax.set_title(fly_stem.split('_')[0], fontsize=9)
    ax.set_xlabel('Factor index')
    if ax is axes[0]:
        ax.set_yticks(range(N_REGIONS))
        ax.set_yticklabels(REGION_NAMES, fontsize=6)
        ax.set_ylabel('Brain region')
    ax.set_xticks(range(0, n_fac, 5))

plt.colorbar(im, ax=axes[-1], label='Norm. readout weight', fraction=0.03)
fig.suptitle('Fig 1 — Readout weight matrix W (regions × factors)', fontsize=11)
plt.tight_layout()
plt.show()
print('Rows = brain regions, Cols = latent factors.\nRed = positive weight, Blue = negative.')

## Fig 2 — Top regions per factor

For each factor, which regions have the largest absolute readout weight?
Averaged across flies using only regions valid in all flies.

In [ ]:
# Average |W| across flies (use only regions valid in all flies)
all_valid = np.ones(N_REGIONS, dtype=bool)
for fly_stem in fly_stems:
    all_valid &= valid_mask[fly_stem]

W_abs_mean = np.zeros((N_REGIONS, n_fac))
for fly_stem in fly_stems:
    W = readout_weights[fly_stem]
    # Normalize per factor before averaging so scale differences don't dominate
    W_norm = np.abs(W) / (np.abs(W).max(axis=0, keepdims=True) + 1e-12)
    W_abs_mean += W_norm
W_abs_mean /= len(fly_stems)
W_abs_mean[~all_valid] = 0.0

TOP_N = 5
top_regions_per_factor = {}
for j in range(n_fac):
    order = np.argsort(W_abs_mean[:, j])[::-1]
    top_regions_per_factor[j] = [(REGION_NAMES[r], W_abs_mean[r, j]) for r in order[:TOP_N]]

# Show top 10 factors by their max loading strength
factor_max_loading = W_abs_mean.max(axis=0)
top_factors = np.argsort(factor_max_loading)[::-1][:10]

print(f'Top {TOP_N} regions for highest-loading factors (averaged |W| across {len(fly_stems)} flies):\n')
for j in top_factors:
    pairs = top_regions_per_factor[j]
    names = ', '.join(f'{r} ({w:.2f})' for r, w in pairs)
    print(f'  Factor {j:2d}: {names}')

## Fig 3 — Factor-region Pearson correlation

From posterior samples: how correlated is each factor's time course with each region's
denoised ΔF/F? This is a data-driven complement to the weight matrix (W captures the
linear model; correlation captures the actual co-variance in the inferred trajectories).

In [ ]:
def factor_region_corr(factors, output_params):
    """Pearson r between each factor and each region across all trials×time."""
    n_t, n_tp, n_fac = factors.shape
    _, _, n_reg = output_params.shape
    F = factors.reshape(-1, n_fac)      # (N, n_fac)
    O = output_params.reshape(-1, n_reg) # (N, n_reg)
    # Z-score both
    F_z = (F - F.mean(0)) / (F.std(0) + 1e-12)
    O_z = (O - O.mean(0)) / (O.std(0) + 1e-12)
    return (F_z.T @ O_z) / len(F)       # (n_fac, n_reg)

fig, axes = plt.subplots(1, n_flies, figsize=(4 * n_flies, 9), sharey=True)
if n_flies == 1:
    axes = [axes]

for ax, fly_stem in zip(axes, fly_stems):
    d = fly_data[fly_stem]
    corr = factor_region_corr(d['factors'], d['output_params'])  # (n_fac, n_reg)
    im = ax.imshow(
        corr.T,  # (n_reg, n_fac)
        aspect='auto', origin='lower',
        cmap='RdBu_r', vmin=-1, vmax=1,
        interpolation='nearest'
    )
    ax.set_title(fly_stem.split('_')[0], fontsize=9)
    ax.set_xlabel('Factor index')
    if ax is axes[0]:
        ax.set_yticks(range(N_REGIONS))
        ax.set_yticklabels(REGION_NAMES, fontsize=6)
        ax.set_ylabel('Brain region')
    ax.set_xticks(range(0, n_fac, 5))

plt.colorbar(im, ax=axes[-1], label='Pearson r', fraction=0.03)
fig.suptitle('Fig 3 — Factor–region Pearson correlation', fontsize=11)
plt.tight_layout()
plt.show()

## Fig 4 — Variance contribution per region per factor

The reconstructed signal for region `i` is `x̂_i = sum_j W[i,j] * f_j + b_i`.
The variance contributed by factor `j` to region `i` is `W[i,j]^2 * Var(f_j)`.
Normalising by total reconstruction variance gives a "factor composition pie" per region.

In [ ]:
fig, axes = plt.subplots(1, n_flies, figsize=(5 * n_flies, 9), sharey=True)
if n_flies == 1:
    axes = [axes]

for ax, fly_stem in zip(axes, fly_stems):
    W = readout_weights[fly_stem]                          # (n_reg, n_fac)
    d = fly_data[fly_stem]
    fac = d['factors'].reshape(-1, W.shape[1])            # (N, n_fac)
    fac_var = fac.var(axis=0)                             # (n_fac,)
    
    # variance contribution matrix: (n_reg, n_fac)
    var_contrib = (W ** 2) * fac_var[np.newaxis, :]       # broadcast over regions
    total_var = var_contrib.sum(axis=1, keepdims=True)    # (n_reg, 1)
    frac = var_contrib / (total_var + 1e-12)              # (n_reg, n_fac)
    
    # Show only top-20 factors by mean explained variance for readability
    top_fac_idx = np.argsort(frac.mean(axis=0))[::-1][:20]
    frac_top = frac[:, top_fac_idx]
    
    im = ax.imshow(
        frac_top,
        aspect='auto', origin='lower',
        cmap='YlOrRd', vmin=0, vmax=frac_top.max(),
        interpolation='nearest'
    )
    ax.set_title(fly_stem.split('_')[0], fontsize=9)
    ax.set_xlabel(f'Top-20 factors (by mean region var. explained)')
    ax.set_xticks(range(20))
    ax.set_xticklabels([str(j) for j in top_fac_idx], rotation=90, fontsize=7)
    if ax is axes[0]:
        ax.set_yticks(range(N_REGIONS))
        ax.set_yticklabels(REGION_NAMES, fontsize=6)
        ax.set_ylabel('Brain region')

plt.colorbar(im, ax=axes[-1], label='Fraction of recon variance', fraction=0.03)
fig.suptitle('Fig 4 — Variance contribution per region per factor', fontsize=11)
plt.tight_layout()
plt.show()

## Fig 5 — Per-region reconstruction R²

How well do the 50 factors reconstruct each region's denoised ΔF/F?
Because the readout is linear, R² = fraction of variance in `output_params`
explained by `factors @ W.T + b` — which should be near 1.0 for all regions
(the readout is trained to minimise this residual).

Low R² for a region means its activity is poorly captured by the shared factor space.

In [ ]:
def per_region_r2(factors, output_params, W, b=None):
    """R² of linear reconstruction per region. W: (n_reg, n_fac)."""
    n_t, n_tp, n_fac = factors.shape
    F = factors.reshape(-1, n_fac)
    O = output_params.reshape(-1, output_params.shape[-1])
    pred = F @ W.T
    if b is not None:
        pred += b[np.newaxis, :]
    else:
        pred += O.mean(axis=0, keepdims=True)  # OLS fit includes bias
    ss_res = ((O - pred) ** 2).sum(axis=0)
    ss_tot = ((O - O.mean(axis=0)) ** 2).sum(axis=0)
    return 1 - ss_res / (ss_tot + 1e-12)  # (n_reg,)

fig, axes = plt.subplots(1, n_flies, figsize=(3 * n_flies, 8), sharey=True)
if n_flies == 1:
    axes = [axes]

for ax, fly_stem in zip(axes, fly_stems):
    W = readout_weights[fly_stem]
    d = fly_data[fly_stem]
    r2 = per_region_r2(d['factors'], d['output_params'], W)
    
    colors = ['steelblue' if v else 'lightgrey' for v in valid_mask[fly_stem]]
    ax.barh(range(N_REGIONS), r2, color=colors, height=0.8)
    ax.axvline(0.8, color='red', lw=0.8, ls='--', label='R²=0.8')
    ax.set_xlim(-0.05, 1.05)
    ax.set_title(fly_stem.split('_')[0], fontsize=9)
    ax.set_xlabel('R²')
    if ax is axes[0]:
        ax.set_yticks(range(N_REGIONS))
        ax.set_yticklabels(REGION_NAMES, fontsize=6)
        ax.set_ylabel('Brain region')
    ax.legend(fontsize=7)

fig.suptitle('Fig 5 — Per-region reconstruction R² (factors → region)', fontsize=11)
plt.tight_layout()
plt.show()
print('Grey bars = regions with near-zero variance (imputed).')

## Fig 6 — Readin vs. readout weight comparison (requires checkpoint)

**Readin** (frozen PCR): `W_in [n_fac × n_regions]` — how much each region contributes
to each latent dimension entering the encoder.

**Readout** (trained): `W_out [n_regions × n_fac]` — how much each factor drives each region
in the reconstruction.

If readout ≈ pinv(readin), training barely moved from PCR initialization.
Divergence = the model learned fly-specific corrections beyond PCR alignment.

In [ ]:
if not readin_weights:
    print('Readin weights require CKPT_PATH to be set. Skipping.')
else:
    fig, axes = plt.subplots(2, n_flies, figsize=(4 * n_flies, 12), sharey='row')
    if n_flies == 1:
        axes = axes[:, np.newaxis]

    for col, fly_stem in enumerate(fly_stems):
        W_in  = readin_weights[fly_stem].T   # transpose → (n_reg, n_fac) for same shape
        W_out = readout_weights[fly_stem]    # (n_reg, n_fac)

        for row, (W, title) in enumerate([
            (W_in / (np.abs(W_in).max(0, keepdims=True) + 1e-12), 'Readin (transposed, norm.)'),
            (W_out / (np.abs(W_out).max(0, keepdims=True) + 1e-12), 'Readout (norm.)'),
        ]):
            ax = axes[row, col]
            im = ax.imshow(W, aspect='auto', origin='lower',
                           cmap='RdBu_r', vmin=-1, vmax=1)
            ax.set_title(f'{fly_stem.split("_")[0]} — {title}', fontsize=8)
            ax.set_xlabel('Factor index')
            if col == 0:
                ax.set_yticks(range(N_REGIONS))
                ax.set_yticklabels(REGION_NAMES, fontsize=6)

    plt.colorbar(im, ax=axes[-1, -1], label='Norm. weight', fraction=0.05)
    fig.suptitle('Fig 6 — Readin (encoder) vs. Readout (decoder) weights', fontsize=11)
    plt.tight_layout()
    plt.show()

## Fig 7 — Cross-fly consistency of region loadings

Are the same regions consistently important across flies?
For each region, compute its mean absolute readout weight norm across factors,
then compare this scalar "region importance" across flies.

In [ ]:
# Region importance per fly: mean |W| across factors (after normalising W columns)
importance = {}  # fly_stem -> (n_regions,)
for fly_stem in fly_stems:
    W = readout_weights[fly_stem]
    W_norm = np.abs(W) / (np.abs(W).max(axis=0, keepdims=True) + 1e-12)
    importance[fly_stem] = W_norm.mean(axis=1)  # (n_regions,)

imp_matrix = np.stack(list(importance.values()), axis=1)  # (n_regions, n_flies)
mean_importance = imp_matrix.mean(axis=1)
std_importance  = imp_matrix.std(axis=1)

# Sort regions by mean importance descending
order = np.argsort(mean_importance)[::-1]
sorted_names = [REGION_NAMES[i] for i in order]
sorted_imp   = imp_matrix[order, :]

fig, axes = plt.subplots(1, 2, figsize=(14, 9))

# Left: heatmap of importance per fly
ax = axes[0]
im = ax.imshow(sorted_imp, aspect='auto', origin='lower', cmap='Oranges')
ax.set_yticks(range(N_REGIONS))
ax.set_yticklabels(sorted_names, fontsize=6)
ax.set_xticks(range(n_flies))
ax.set_xticklabels([s.split('_')[0] for s in fly_stems], fontsize=8, rotation=45)
ax.set_xlabel('Fly')
ax.set_ylabel('Brain region (sorted by mean importance)')
ax.set_title('Region importance per fly', fontsize=9)
plt.colorbar(im, ax=ax, label='Mean |W| (norm.)', fraction=0.04)

# Right: mean ± std across flies
ax2 = axes[1]
y = range(N_REGIONS)
ax2.barh(y, mean_importance[order], xerr=std_importance[order],
         color='steelblue', height=0.7, capsize=2, ecolor='grey')
ax2.set_yticks(y)
ax2.set_yticklabels(sorted_names, fontsize=6)
ax2.set_xlabel('Mean |W| across flies (norm.)')
ax2.set_title('Mean region importance ± std', fontsize=9)

fig.suptitle('Fig 7 — Cross-fly region importance', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Top 10 most consistently loaded regions:')
for i in range(10):
    r = order[i]
    print(f'  {REGION_NAMES[r]:12s}  mean={mean_importance[r]:.3f}  std={std_importance[r]:.3f}')

## Notes and caveats

- **Rotation ambiguity**: factor indices have no canonical meaning — LFADS finds a
  rotation-free subspace. Individual factor numbers change across runs.
  For stable interpretability, rotate factors to stimulus directions (e.g., PCA of
  condition-mean factors) before reading off per-factor region loadings.

- **PCR initialization**: readout weights start as `pinv(W_readin)` (see
  `how_multisession_lfads.md`). Training refines from there. If training converged
  quickly, readout ≈ pinv(readin) and both reflect the PCR alignment, not purely
  neural structure.

- **18 trials per fly**: with limited data, variational inference may be poorly
  constrained. Treat loadings as indicative rather than quantitatively precise.

- **Factors → condition directions**: for the cleanest region-level story, project
  factors onto stimulus PCs (e.g., odour axis, walking axis) first, then ask which
  regions load onto those named axes.